# Stage 00 – Archive Listing & Manifest Creation
**Clean Medical-Grade Edition — Report 2 Synced**

Mục tiêu:
- Không giải nén toàn bộ PET/CT image ở Stage 00.
- Chỉ dùng `7z l -slt` để quét archive.
- Chỉ extract JSON report để parse Findings/Impression.
- Tạo manifest an toàn cho Stage 01.
- Kiểm tra patient-level leakage, section completeness, keyword/SUV/token readiness.

> PM Gate: Output Stage 00 phải được review trước khi chạy Stage 01.

In [ ]:
# CELL 1 — Kaggle-only 7z install/check
# Nếu chạy trong Kaggle notebook, cell magic này hợp lệ.
# Nếu chạy file .py local, bỏ qua cell này và đảm bảo máy đã có 7z.
try:
    get_ipython  # noqa: F821
    get_ipython().system("which 7z || (apt-get update -qq && apt-get install -y -qq p7zip-full)")
except NameError:
    pass

In [ ]:
# CELL 2 — Imports + Configuration
from pathlib import Path
from collections import Counter, defaultdict
import os
import re
import json
import shutil
import hashlib
import subprocess
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 200)

IS_KAGGLE = Path("/kaggle").exists()
OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if IS_KAGGLE else Path("vimedpet_q1_outputs")
MANIFEST_DIR = OUTPUT_ROOT / "00_manifest"
TABLE_DIR = MANIFEST_DIR / "tables"
FIGURE_DIR = MANIFEST_DIR / "figures"

SCRATCH_ROOT = Path("/kaggle/temp/vimedpet_stage00") if IS_KAGGLE else OUTPUT_ROOT / "_scratch_stage00"
STAGE_ROOT = SCRATCH_ROOT / "archive_stage"
JSON_ROOT = SCRATCH_ROOT / "report_json"

for p in [MANIFEST_DIR, TABLE_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if SCRATCH_ROOT.exists():
    shutil.rmtree(SCRATCH_ROOT, ignore_errors=True)
for p in [SCRATCH_ROOT, STAGE_ROOT, JSON_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Keyword list từ Report 2.
CLINICAL_KEYWORDS = [
    "hạch", "phổi", "xương", "thận", "gan", "cổ", "chậu", "đại tràng", "ổ bụng", "lách",
    "não", "amydal", "trung thất", "tuyến giáp", "dạ dày", "vú", "thượng đòn", "vòm", "tụy",
    "tuyến tiền liệt",
]

TOKEN_PROXY_CANDIDATES = [256, 384, 512, 768]
RECOMMENDED_MAX_SEQ_LENGTH = 512

print("IS_KAGGLE:", IS_KAGGLE)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("MANIFEST_DIR:", MANIFEST_DIR)
print("SCRATCH_ROOT:", SCRATCH_ROOT)

In [ ]:
# CELL 3 — Discover archive markers
INPUT_HINTS = [
    # Kaggle Notebook UI thường mount dataset theo dạng /kaggle/input/<dataset-slug>.
    Path("/kaggle/input/vimed-pet-ct-part1-raw-no-autoextract-v3"),
    Path("/kaggle/input/vimed-pet-ct-part2-raw-no-autoextract-v1"),
    Path("/kaggle/input/vimed-pet-ct-part3-raw-no-autoextract-v1"),
    # Fallback nếu môi trường có cấu trúc Kaggle API/cache dạng owner/dataset.
    Path("/kaggle/input/datasets/tundng111/vimed-pet-ct-part1-raw-no-autoextract-v3"),
    Path("/kaggle/input/datasets/tundng111/vimed-pet-ct-part2-raw-no-autoextract-v1"),
    Path("/kaggle/input/datasets/tundng111/vimed-pet-ct-part3-raw-no-autoextract-v1"),
    # Fallback cuối: quét toàn bộ /kaggle/input để không phụ thuộc tên mount.
    Path("/kaggle/input"),
    Path.cwd(),
]

def existing_roots():
    roots, seen = [], set()
    for root in INPUT_HINTS:
        if root.exists():
            rr = root.resolve()
            if rr not in seen:
                roots.append(rr)
                seen.add(rr)
    return roots

def archive_stem_from_marker(marker: Path) -> str:
    name = marker.name
    if name.endswith(".zipraw"):
        return name[:-len(".zipraw")]
    if name.endswith(".zip"):
        return name[:-len(".zip")]
    return marker.stem

roots = existing_roots()
print("Found roots:")
for r in roots:
    print(" -", r)

all_markers = []
for root in roots:
    all_markers.extend(root.rglob("*.zipraw"))
    all_markers.extend(root.rglob("*.zip"))
all_markers = sorted(set(all_markers))

archive_groups = {}
for marker in all_markers:
    stem = archive_stem_from_marker(marker)
    # Ưu tiên .zipraw nếu cùng stem có cả .zip và .zipraw.
    if stem in archive_groups and marker.suffix != ".zipraw":
        continue
    parts = sorted(marker.parent.glob(f"{stem}.z[0-9][0-9]"))
    archive_groups[stem] = {
        "stem": stem,
        "folder": marker.parent,
        "marker": marker,
        "parts": parts,
    }

ARCHIVE_STEMS = sorted(archive_groups)
print("\nArchive groups:", len(ARCHIVE_STEMS))
for stem in ARCHIVE_STEMS:
    g = archive_groups[stem]
    print(f"- {stem}: marker={g['marker'].name}, parts={len(g['parts'])}")

assert ARCHIVE_STEMS, "Không tìm thấy .zipraw/.zip. Hãy Add Data Part 1/2/3 trên Kaggle."

In [ ]:
# CELL 4 — 7z helpers

def sevenzip_binary():
    for exe in ["7z", "7zz", "7za"]:
        try:
            subprocess.run([exe], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return exe
        except Exception:
            pass
    raise RuntimeError("Không tìm thấy 7z/7zz/7za.")

SEVENZIP = sevenzip_binary()
print("Using:", SEVENZIP)

def safe_unlink(path: Path):
    if path.exists() or path.is_symlink():
        path.unlink()

def symlink_required(src: Path, dst: Path):
    safe_unlink(dst)
    os.symlink(src, dst)

def prepare_archive_stage(stem: str) -> Path:
    g = archive_groups[stem]
    stage = STAGE_ROOT / stem
    if stage.exists():
        shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir(parents=True, exist_ok=True)

    for p in g["parts"]:
        symlink_required(p, stage / p.name)

    zip_path = stage / f"{stem}.zip"
    symlink_required(g["marker"], zip_path)
    return zip_path

def cleanup_archive_stage(stem: str):
    stage = STAGE_ROOT / stem
    if stage.exists():
        shutil.rmtree(stage, ignore_errors=True)

def run_7z_list(zip_path: Path) -> str:
    cmd = [SEVENZIP, "l", "-slt", str(zip_path)]
    r = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.returncode != 0:
        raise RuntimeError(
            "7z list failed\n"
            f"CMD: {' '.join(cmd)}\n"
            f"STDERR:\n{r.stderr[:4000]}"
        )
    return r.stdout

def run_7z_extract_specific(zip_path: Path, out_dir: Path, inner_paths):
    inner_paths = [str(p) for p in inner_paths if str(p).strip()]
    if not inner_paths:
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [SEVENZIP, "x", str(zip_path), f"-o{out_dir}", "-y"] + inner_paths
    r = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.returncode != 0:
        raise RuntimeError(
            "7z extract failed\n"
            f"CMD prefix: {' '.join(cmd[:6])} ... n_files={len(inner_paths)}\n"
            f"STDERR:\n{r.stderr[:4000]}"
        )

In [ ]:
# CELL 5 — Archive inventory listing (no image extraction)

def parse_7z_slt(text: str, source_part: str):
    rows = []
    for block in text.split("\n\n"):
        info = {}
        for line in block.splitlines():
            if " = " in line:
                k, v = line.split(" = ", 1)
                info[k.strip()] = v.strip()
        path = info.get("Path")
        if not path:
            continue
        attr = info.get("Attributes", "")
        if attr.startswith("D"):
            continue
        try:
            size = int(info.get("Size", 0))
        except Exception:
            size = 0
        rows.append({
            "source_part": source_part,
            "archive_path": path.replace("\\", "/"),
            "size": size,
            "attributes": attr,
        })
    return rows

all_rows = []
for stem in ARCHIVE_STEMS:
    print(f"Listing archive: {stem}")
    zip_path = prepare_archive_stage(stem)
    try:
        text = run_7z_list(zip_path)
        rows = parse_7z_slt(text, source_part=stem)
        print(" files:", len(rows))
        all_rows.extend(rows)
    finally:
        cleanup_archive_stage(stem)

inventory = pd.DataFrame(all_rows)
assert len(inventory) > 0, "Archive listing rỗng. Kiểm tra lại dataset input."
inventory.to_csv(MANIFEST_DIR / "q1_archive_inventory.csv", index=False, encoding="utf-8-sig")
print("Total inventory rows:", len(inventory))
display(inventory.head())

In [ ]:
# CELL 6 — Path parsing and modality detection

def norm_path(p) -> str:
    return str(p).replace("\\", "/")

def lower_path(p) -> str:
    return "/" + norm_path(p).lower().strip("/")

def is_npy(p) -> bool:
    return lower_path(p).endswith(".npy")

def is_json(p) -> bool:
    return lower_path(p).endswith(".json")

def is_ct_path(p) -> bool:
    s = lower_path(p)
    return is_npy(s) and ("/ct/" in s or "/ct_" in s or "ct/" in s)

def is_pet_path(p) -> bool:
    s = lower_path(p)
    return is_npy(s) and ("/pet/" in s or "/pet_" in s or "pet/" in s)

def patient_id_from_path(p) -> str:
    s = norm_path(p)
    patterns = [
        r"patient[_\s-]*(\d+)",
        r"benh[_\s-]*nhan[_\s-]*(\d+)",
        r"bn[_\s-]*(\d+)",
    ]
    for pat in patterns:
        m = re.search(pat, s, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return "unknown"

def month_from_path(p) -> int:
    s = norm_path(p)
    patterns = [r"THANG\s*(\d+)", r"thang[_\s-]*(\d+)", r"month[_\s-]*(\d+)"]
    for pat in patterns:
        m = re.search(pat, s, flags=re.IGNORECASE)
        if m:
            return int(m.group(1))
    return -1

def day_from_path(p) -> int:
    s = norm_path(p)
    patterns = [r"day[_\s-]*(\d+)", r"ngay[_\s-]*(\d+)"]
    for pat in patterns:
        m = re.search(pat, s, flags=re.IGNORECASE)
        if m:
            return int(m.group(1))
    return -1

def parent_exam_key(p) -> str:
    """Key phụ giúp không gộp nhầm multi-exam cùng patient."""
    parts = Path(norm_path(p)).parts
    if len(parts) >= 2:
        return "/".join(parts[:-1])
    return norm_path(p)

def stable_id(*items) -> str:
    key = "|".join(map(str, items)).encode("utf-8")
    return hashlib.sha1(key).hexdigest()[:12]

inventory["archive_path"] = inventory["archive_path"].map(norm_path)
inventory["is_ct"] = inventory["archive_path"].map(is_ct_path)
inventory["is_pet"] = inventory["archive_path"].map(is_pet_path)
inventory["is_report"] = inventory["archive_path"].map(is_json)
inventory["patient_key"] = inventory["archive_path"].map(patient_id_from_path)
inventory["month"] = inventory["archive_path"].map(month_from_path)
inventory["day"] = inventory["archive_path"].map(day_from_path)
inventory["exam_key"] = inventory["archive_path"].map(parent_exam_key)

print("CT files:", int(inventory["is_ct"].sum()))
print("PET files:", int(inventory["is_pet"].sum()))
print("JSON reports:", int(inventory["is_report"].sum()))
display(inventory[inventory["is_report"]].head())

In [ ]:
# CELL 7 — Pair CT/PET/Report cases safely
# Ưu tiên ghép theo source_part + patient + month + day + exam-folder gần nhất.
# Nếu cấu trúc folder không đồng nhất, fallback vẫn giữ candidate count để QA.

case_buckets = defaultdict(lambda: {"ct_paths": [], "pet_paths": [], "report_paths": []})

for _, row in inventory.iterrows():
    if not (row["is_ct"] or row["is_pet"] or row["is_report"]):
        continue
    key = (
        row["source_part"],
        str(row["patient_key"]),
        int(row["month"]),
        int(row["day"]),
        str(row["exam_key"]),
    )
    if row["is_ct"]:
        case_buckets[key]["ct_paths"].append(row["archive_path"])
    elif row["is_pet"]:
        case_buckets[key]["pet_paths"].append(row["archive_path"])
    elif row["is_report"]:
        case_buckets[key]["report_paths"].append(row["archive_path"])

# Fallback: nếu folder-level quá tách biệt làm CT/PET/report không vào cùng bucket,
# ghép lại ở cấp source_part + patient + month + day.
coarse_buckets = defaultdict(lambda: {"ct_paths": [], "pet_paths": [], "report_paths": [], "exam_keys": set()})
for key, bucket in case_buckets.items():
    source_part, patient_key, month, day, exam_key = key
    coarse_key = (source_part, patient_key, month, day)
    coarse_buckets[coarse_key]["ct_paths"].extend(bucket["ct_paths"])
    coarse_buckets[coarse_key]["pet_paths"].extend(bucket["pet_paths"])
    coarse_buckets[coarse_key]["report_paths"].extend(bucket["report_paths"])
    coarse_buckets[coarse_key]["exam_keys"].add(exam_key)

case_rows, unpaired_rows = [], []
for key, bucket in coarse_buckets.items():
    source_part, patient_key, month, day = key
    ct_paths = sorted(set(bucket["ct_paths"]))
    pet_paths = sorted(set(bucket["pet_paths"]))
    report_paths = sorted(set(bucket["report_paths"]))

    if ct_paths and pet_paths and report_paths:
        n_cases = max(len(ct_paths), len(pet_paths), len(report_paths))
        for i in range(n_cases):
            ct_path = ct_paths[min(i, len(ct_paths) - 1)]
            pet_path = pet_paths[min(i, len(pet_paths) - 1)]
            report_path = report_paths[min(i, len(report_paths) - 1)]
            case_rows.append({
                "source_part": source_part,
                "patient_key": patient_key,
                "month": month,
                "day": day,
                "ct_path": ct_path,
                "pet_path": pet_path,
                "report_path": report_path,
                "n_ct_candidates": len(ct_paths),
                "n_pet_candidates": len(pet_paths),
                "n_report_candidates": len(report_paths),
                "case_id": stable_id(source_part, patient_key, month, day, i, ct_path, pet_path, report_path),
            })
    else:
        unpaired_rows.append({
            "source_part": source_part,
            "patient_key": patient_key,
            "month": month,
            "day": day,
            "n_ct": len(ct_paths),
            "n_pet": len(pet_paths),
            "n_report": len(report_paths),
        })

manifest = pd.DataFrame(case_rows).drop_duplicates("case_id").reset_index(drop=True)
unpaired = pd.DataFrame(unpaired_rows)

print("Paired cases before text parsing:", len(manifest))
print("Unpaired buckets:", len(unpaired))
display(manifest.head())
if len(unpaired):
    display(unpaired.head())
unpaired.to_csv(MANIFEST_DIR / "q1_unpaired_buckets.csv", index=False, encoding="utf-8-sig")

assert len(manifest) > 0, "Không ghép được case CT/PET/Report nào. Cần kiểm tra path patterns."

In [ ]:
# CELL 8 — Extract JSON reports only
print("Extracting JSON reports only...")

reports_by_stem = {
    stem: manifest.loc[manifest["source_part"] == stem, "report_path"].dropna().astype(str).tolist()
    for stem in sorted(manifest["source_part"].unique())
}

for stem, paths in reports_by_stem.items():
    if not paths:
        continue
    print(f"Archive {stem}: extracting {len(paths)} JSON reports")
    zip_path = prepare_archive_stage(stem)
    try:
        batch_size = 300
        for i in range(0, len(paths), batch_size):
            batch = paths[i:i + batch_size]
            print(f"  batch {i // batch_size + 1}: {len(batch)}")
            run_7z_extract_specific(zip_path, JSON_ROOT, batch)
    finally:
        cleanup_archive_stage(stem)

print("JSON_ROOT:", JSON_ROOT)
print("Extracted JSON files:", len(list(JSON_ROOT.rglob("*.json"))))

In [ ]:
# CELL 9 — Robust JSON report parser

def clean_text(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return re.sub(r"\s+", " ", str(x).replace("\u00a0", " ")).strip()

def normalize_key(k) -> str:
    text = clean_text(k).lower()
    text = text.replace("đ", "d")
    # Không dùng unidecode để tránh thêm dependency; xử lý các dấu chính qua map đơn giản.
    repl = {
        "áàảãạăắằẳẵặâấầẩẫậ": "a",
        "éèẻẽẹêếềểễệ": "e",
        "íìỉĩị": "i",
        "óòỏõọôốồổỗộơớờởỡợ": "o",
        "úùủũụưứừửữự": "u",
        "ýỳỷỹỵ": "y",
    }
    for chars, repl_char in repl.items():
        for ch in chars:
            text = text.replace(ch, repl_char)
    text = re.sub(r"[^a-z0-9]+", " ", text).strip()
    return text

def get_first_value(obj, keys):
    if not isinstance(obj, dict):
        return ""
    lookup = {normalize_key(k): v for k, v in obj.items()}
    for key in keys:
        nk = normalize_key(key)
        if nk in lookup:
            return lookup[nk]
    return ""

def read_json_safely(path: Path):
    for enc in ["utf-8", "utf-8-sig"]:
        try:
            return json.loads(path.read_text(encoding=enc, errors="replace")), ""
        except Exception as exc:
            last_error = str(exc)
    return {}, last_error

def parse_report_obj(obj):
    if not isinstance(obj, dict):
        raw = clean_text(obj)
        return {
            "findings": raw,
            "impression": "",
            "report_text_clean": raw,
            "sex": "",
            "birth_year": "",
            "indication": "",
            "clinical_history": "",
            "parse_warning": "non_dict_json",
        }

    findings_obj = get_first_value(obj, [
        "Mô tả hình ảnh", "Mo ta hinh anh", "Mô tả", "Mo ta", "Findings", "FINDINGS",
    ])
    impression_obj = get_first_value(obj, [
        "Nhận định kết quả", "Nhan dinh ket qua", "Nhận định", "Nhan dinh",
        "Kết luận", "Ket luan", "Impression", "IMPRESSION",
    ])

    finding_parts = []
    if isinstance(findings_obj, dict):
        for section, value in findings_obj.items():
            section_text = clean_text(section)
            value_text = clean_text(value)
            if value_text:
                finding_parts.append(f"{section_text}: {value_text}" if section_text else value_text)
    else:
        value_text = clean_text(findings_obj)
        if value_text:
            finding_parts.append(value_text)
    findings = "\n".join(finding_parts).strip()
    impression = clean_text(impression_obj)

    sex = clean_text(get_first_value(obj, ["Giới tính", "Gioi tinh", "Sex", "Gender"]))
    birth_year = clean_text(get_first_value(obj, ["Năm sinh", "Nam sinh", "Birth year", "Year of birth"]))
    indication = clean_text(get_first_value(obj, ["Chỉ định", "Chi dinh", "Indication"]))
    clinical_history = clean_text(get_first_value(obj, [
        "Bệnh sử lâm sàng", "Benh su lam sang", "Bệnh sử", "Benh su", "Clinical history",
    ]))

    target_parts = []
    if findings:
        target_parts.append("MÔ TẢ HÌNH ẢNH:\n" + findings)
    if impression:
        target_parts.append("NHẬN ĐỊNH KẾT QUẢ:\n" + impression)
    report_text_clean = "\n\n".join(target_parts).strip()

    warnings_list = []
    if not findings:
        warnings_list.append("missing_findings")
    if not impression:
        warnings_list.append("missing_impression")

    return {
        "findings": findings,
        "impression": impression,
        "report_text_clean": report_text_clean,
        "sex": sex,
        "birth_year": birth_year,
        "indication": indication,
        "clinical_history": clinical_history,
        "parse_warning": ";".join(warnings_list),
    }

parsed_rows = []
for _, row in manifest.iterrows():
    local_json = JSON_ROOT / row["report_path"]
    if not local_json.exists():
        parsed_rows.append({
            "findings": "",
            "impression": "",
            "report_text_clean": "",
            "sex": "",
            "birth_year": "",
            "indication": "",
            "clinical_history": "",
            "parse_warning": "json_not_extracted",
            "parse_error": "json_not_extracted",
        })
        continue
    obj, err = read_json_safely(local_json)
    parsed = parse_report_obj(obj) if not err else {
        "findings": "",
        "impression": "",
        "report_text_clean": "",
        "sex": "",
        "birth_year": "",
        "indication": "",
        "clinical_history": "",
        "parse_warning": "json_parse_error",
    }
    parsed["parse_error"] = err
    parsed_rows.append(parsed)

parsed_df = pd.DataFrame(parsed_rows)
manifest = pd.concat([manifest.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)

manifest["has_findings"] = manifest["findings"].fillna("").astype(str).str.strip().ne("")
manifest["has_impression"] = manifest["impression"].fillna("").astype(str).str.strip().ne("")
manifest["parse_ok"] = manifest["parse_error"].fillna("").astype(str).str.strip().eq("")
manifest["report_word_count"] = manifest["report_text_clean"].fillna("").astype(str).map(lambda x: len(x.split()))
manifest["report_char_count"] = manifest["report_text_clean"].fillna("").astype(str).map(len)
manifest["usable_for_training"] = (
    manifest["parse_ok"]
    & manifest["has_findings"]
    & manifest["has_impression"]
    & manifest["report_text_clean"].fillna("").astype(str).str.len().gt(10)
)

print("Parsed OK:", int(manifest["parse_ok"].sum()), "/", len(manifest))
print("Has findings:", int(manifest["has_findings"].sum()), "/", len(manifest))
print("Has impression:", int(manifest["has_impression"].sum()), "/", len(manifest))
print("Usable for training:", int(manifest["usable_for_training"].sum()), "/", len(manifest))
display(manifest[["case_id", "patient_key", "has_findings", "has_impression", "parse_warning", "report_word_count"]].head())

# Không xóa JSON_ROOT cho đến cuối để debug nếu cần.

In [ ]:
# CELL 10 — Patient-level split (70/15/15, no leakage)
clean_manifest = manifest[manifest["usable_for_training"]].copy().reset_index(drop=True)

patients = sorted(clean_manifest["patient_key"].astype(str).unique().tolist())
train_end = int(round(len(patients) * 0.70))
val_end = int(round(len(patients) * 0.85))

split_map = {}
for i, p in enumerate(patients):
    if i < train_end:
        split_map[p] = "train"
    elif i < val_end:
        split_map[p] = "validation"
    else:
        split_map[p] = "test"

clean_manifest["clean_split"] = clean_manifest["patient_key"].astype(str).map(split_map)
manifest["clean_split"] = manifest["patient_key"].astype(str).map(split_map).fillna("excluded")

def patient_overlap(df):
    by = {
        s: set(df.loc[df["clean_split"] == s, "patient_key"].astype(str))
        for s in ["train", "validation", "test"]
    }
    return {
        "train_validation": len(by["train"] & by["validation"]),
        "train_test": len(by["train"] & by["test"]),
        "validation_test": len(by["validation"] & by["test"]),
    }

overlap = patient_overlap(clean_manifest)
print("Clean cases:", len(clean_manifest))
print("Unique patients:", clean_manifest["patient_key"].nunique())
print("Split counts:")
print(clean_manifest["clean_split"].value_counts().to_string())
print("Patient overlap:", overlap)
assert all(v == 0 for v in overlap.values()), f"Patient leakage detected: {overlap}"

In [ ]:
# CELL 11 — Clinical QA from Report 2
section_completeness = pd.DataFrame([
    {
        "field": "findings",
        "present": int(manifest["has_findings"].sum()),
        "total": int(len(manifest)),
        "coverage_percent": float(manifest["has_findings"].mean() * 100) if len(manifest) else 0.0,
    },
    {
        "field": "impression",
        "present": int(manifest["has_impression"].sum()),
        "total": int(len(manifest)),
        "coverage_percent": float(manifest["has_impression"].mean() * 100) if len(manifest) else 0.0,
    },
])
section_completeness.to_csv(TABLE_DIR / "01_section_completeness.csv", index=False, encoding="utf-8-sig")
display(section_completeness)

full_text = "\n".join(clean_manifest["report_text_clean"].fillna("").astype(str)).lower()
keyword_counts = Counter({kw: full_text.count(kw) for kw in CLINICAL_KEYWORDS})
keyword_df = pd.DataFrame(keyword_counts.most_common(), columns=["keyword", "count"])
keyword_df.to_csv(TABLE_DIR / "02_keyword_counts.csv", index=False, encoding="utf-8-sig")
display(keyword_df)

def suv_count(text):
    t = str(text).lower()
    return len(re.findall(r"\bsuv(?:max)?\b|suv\s*max", t, flags=re.IGNORECASE))

def extract_suv_values(text):
    vals = []
    for m in re.finditer(r"suv(?:\s*max)?\s*[:=]?\s*([0-9]+(?:[.,][0-9]+)?)", str(text).lower(), flags=re.IGNORECASE):
        try:
            vals.append(float(m.group(1).replace(",", ".")))
        except Exception:
            pass
    return vals

clean_manifest["suv_mention_count"] = clean_manifest["report_text_clean"].map(suv_count)
manifest["suv_mention_count"] = manifest["report_text_clean"].fillna("").map(suv_count)
all_suv_values = []
for t in clean_manifest["report_text_clean"]:
    all_suv_values.extend(extract_suv_values(t))

suv_stats = {
    "reports_with_suv": int((clean_manifest["suv_mention_count"] > 0).sum()),
    "total_reports": int(len(clean_manifest)),
    "suv_values_parsed": int(len(all_suv_values)),
    "suv_value_mean": float(np.mean(all_suv_values)) if all_suv_values else None,
    "suv_value_median": float(np.median(all_suv_values)) if all_suv_values else None,
    "suv_value_max": float(np.max(all_suv_values)) if all_suv_values else None,
}
pd.DataFrame([suv_stats]).to_csv(TABLE_DIR / "03_suv_stats.csv", index=False, encoding="utf-8-sig")
display(pd.DataFrame([suv_stats]))

token_rows = []
for max_tokens in TOKEN_PROXY_CANDIDATES:
    covered = int((clean_manifest["report_word_count"] <= max_tokens).sum())
    token_rows.append({
        "max_tokens_proxy": max_tokens,
        "reports_covered": covered,
        "total_reports": int(len(clean_manifest)),
        "coverage_percent": float(covered / max(len(clean_manifest), 1) * 100),
    })
token_readiness = pd.DataFrame(token_rows)
token_readiness.to_csv(TABLE_DIR / "04_max_token_readiness.csv", index=False, encoding="utf-8-sig")
display(token_readiness)

word_stats = clean_manifest["report_word_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).to_frame("report_word_count").T
word_stats.to_csv(TABLE_DIR / "05_report_word_stats.csv", index=False, encoding="utf-8-sig")
display(word_stats)

In [ ]:
# CELL 12 — Export Stage 00 outputs
manifest = manifest.sort_values(["source_part", "patient_key", "month", "day", "case_id"]).reset_index(drop=True)
clean_manifest = clean_manifest.sort_values(["source_part", "patient_key", "month", "day", "case_id"]).reset_index(drop=True)

manifest["row_index_all"] = np.arange(len(manifest), dtype=int)
clean_manifest["row_index"] = np.arange(len(clean_manifest), dtype=int)

all_cases_path = MANIFEST_DIR / "q1_all_cases_manifest_with_flags.csv"
main_manifest_path = MANIFEST_DIR / "q1_clean_manifest.csv"
cache_manifest_path = MANIFEST_DIR / "q1_clean_manifest_for_cache.csv"

manifest.to_csv(all_cases_path, index=False, encoding="utf-8-sig")
clean_manifest.to_csv(main_manifest_path, index=False, encoding="utf-8-sig")
clean_manifest.to_csv(cache_manifest_path, index=False, encoding="utf-8-sig")

summary = {
    "stage": "00_archive_listing_manifest_clean_medical_grade",
    "total_inventory_files": int(len(inventory)),
    "ct_files": int(inventory["is_ct"].sum()),
    "pet_files": int(inventory["is_pet"].sum()),
    "json_reports": int(inventory["is_report"].sum()),
    "paired_cases_before_text_filter": int(len(manifest)),
    "clean_cases": int(len(clean_manifest)),
    "unique_patients_clean": int(clean_manifest["patient_key"].nunique()),
    "split_counts": {k: int(v) for k, v in clean_manifest["clean_split"].value_counts().to_dict().items()},
    "patient_overlap": overlap,
    "has_findings_all_cases": int(manifest["has_findings"].sum()),
    "has_impression_all_cases": int(manifest["has_impression"].sum()),
    "missing_findings_all_cases": int((~manifest["has_findings"]).sum()),
    "missing_impression_all_cases": int((~manifest["has_impression"]).sum()),
    "parse_ok_all_cases": int(manifest["parse_ok"].sum()),
    "mean_report_word_count_clean": float(clean_manifest["report_word_count"].mean()) if len(clean_manifest) else 0.0,
    "median_report_word_count_clean": float(clean_manifest["report_word_count"].median()) if len(clean_manifest) else 0.0,
    "max_report_word_count_clean": int(clean_manifest["report_word_count"].max()) if len(clean_manifest) else 0,
    "recommended_max_seq_length_for_stage03": RECOMMENDED_MAX_SEQ_LENGTH,
    "suv_stats": suv_stats,
}
summary_path = MANIFEST_DIR / "q1_manifest_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

# Dọn JSON scratch sau khi export để tiết kiệm disk.
shutil.rmtree(JSON_ROOT, ignore_errors=True)

print("===== STAGE 00 DONE =====")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nOutput files:")
for p in sorted(MANIFEST_DIR.rglob("*")):
    if p.is_file():
        print(p, p.stat().st_size, "bytes")

In [ ]:
# CELL 13 — PM checkpoint
print("===== PM CHECKPOINT — DO NOT RUN STAGE 01 BEFORE REVIEW =====")
print("Required outputs for Stage 01:")
print("1.", MANIFEST_DIR / "q1_clean_manifest.csv")
print("2.", MANIFEST_DIR / "q1_clean_manifest_for_cache.csv")
print("3.", MANIFEST_DIR / "q1_manifest_summary.json")
print()
print("Acceptance checks:")
print("- patient_overlap must be all zeros:", summary["patient_overlap"])
print("- clean_cases must match the intended dataset scope.")
print("- missing_findings/missing_impression must be reviewed, not silently ignored.")
print("- Stage 03 should use MAX_SEQ_LENGTH=512 unless validation proves otherwise.")
print()
print("Sau khi chạy xong: Save Version trên Kaggle rồi gửi tôi q1_manifest_summary.json để PM review.")